In [1]:
import numpy as np
import random
import os
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import torch.nn as nn
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight

## seeding for reproducibility
def seed_all(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

# configuration
n_qubits = 6  # kept same for fair comparison
batch_size = 64
num_classes = 10
num_epochs = 80
lr = 0.0003

# transforms
train_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

test_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

TRAIN_PATH = 'train'
TEST_PATH = 'test'
VAL_PATH = './val'

try:
    train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
    test_dataset = ImageFolder(TEST_PATH, transform=test_transform)
    print("dataset loaded successfully")
except Exception as e:
    print(f"Error loading datasets: {e}")

try:
    labels = [label for _, label in train_dataset.samples]
    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
    class_weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
except:
    print("could not calculate class weights")
    print("Now using equal weights while training")
    class_weight_tensor = torch.ones(num_classes).to(device)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

val_dataset = ImageFolder(VAL_PATH, transform=eval_transform)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)


# same FeatureReduce as before
class FeatureReduce(nn.Module):
    def __init__(self, final_dim, dropout=0.2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc_expand = nn.Linear(64, final_dim * 2)
        self.fc_project = nn.Linear(final_dim * 2, final_dim)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = self.fc_expand(x)
        x = self.activation(x)
        x = self.fc_project(x)
        return x


# Classical layer replacing the quantum circuit
# Mirrors what the quantum layer does: takes n_qubits inputs -> returns n_qubits outputs
# Parameter count kept comparable: quantum had (3 * n_qubits) weights, so we match that
class ClassicalAnalogLayer(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        # 3 * n_qubits weights to match quantum circuit's weight_shapes = {"weights": (3, n_qubits)}
        self.layer = nn.Sequential(
            nn.Linear(n_features, n_features * 3),  # expand like 3 layers of rotations
            nn.Tanh(),                               # tanh mirrors quantum expectation value range [-1, 1]
            nn.Linear(n_features * 3, n_features)   # project back
        )

    def forward(self, x):
        return self.layer(x)


class ClassicalQNN(nn.Module):
    def __init__(self, n_features, num_classes):
        super().__init__()
        self.feature_extractor = FeatureReduce(final_dim=n_features)
        self.classical_layer = ClassicalAnalogLayer(n_features)  # replaces quantum layer
        self.classifier = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = torch.tanh(x)               # same preprocessing as before quantum layer
        x = self.classical_layer(x)     # classical analog of quantum circuit
        return self.classifier(x)


def train_epoch(model, dataloader, loss_fn, optimizer, device):
    model.train()
    total_loss, correct = 0.0, 0

    for inputs, labels in tqdm(dataloader, desc="Training", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(dim=1) == labels).sum().item()

    return total_loss / len(dataloader), correct / len(dataloader.dataset)


def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return total_loss / len(dataloader), correct / total


# model, optimizer, loss
model = ClassicalQNN(n_features=n_qubits, num_classes=num_classes).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.005)
loss_fn = nn.CrossEntropyLoss(weight=class_weight_tensor)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

best_val_loss = float('inf')
train_losses, val_losses = [], []
train_accs, val_accs = [], []
early_stopping_patience = 15
epochs_without_improvement = 0

print(f"Starting Classical Training for {num_epochs} epochs...")

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, loss_fn, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "exp2_classical.pth")
        print("   💾 Best model saved.")
    else:
        epochs_without_improvement += 1
        print(f"   🕒 No improvement for {epochs_without_improvement} epoch(s).")

    if epochs_without_improvement >= early_stopping_patience:
        print(f"⏹️ Early stopping triggered after {epoch+1} epochs.")
        break

using device: cuda
dataset loaded successfully
Starting Classical Training for 80 epochs...


Epoch 1/80
   Train Loss: 1.8752 | Train Acc: 0.3941 | Val Acc: 0.3980
   💾 Best model saved.


Epoch 2/80
   Train Loss: 1.2029 | Train Acc: 0.5075 | Val Acc: 0.6376
   💾 Best model saved.


Epoch 3/80
   Train Loss: 1.0085 | Train Acc: 0.6232 | Val Acc: 0.7335
   💾 Best model saved.


Epoch 4/80
   Train Loss: 0.9036 | Train Acc: 0.7000 | Val Acc: 0.7881
   💾 Best model saved.


...

Epoch 51/80
   Train Loss: 0.3998 | Train Acc: 0.8650 | Val Acc: 0.8468
   🕒 No improvement for 11 epoch(s).


Epoch 52/80
   Train Loss: 0.4002 | Train Acc: 0.8632 | Val Acc: 0.8253
   🕒 No improvement for 12 epoch(s).


Epoch 53/80
   Train Loss: 0.3937 | Train Acc: 0.8675 | Val Acc: 0.8371
   🕒 No improvement for 13 epoch(s).


Epoch 54/80
   Train Loss: 0.3895 | Train Acc: 0.8649 | Val Acc: 0.8491
   🕒 No improvement for 14 epoch(s).


Epoch 55/80
   Train Loss: 0.3937 | Train Acc: 0.8665 | Val Acc: 0.8389
   🕒 No improvement for 15 epoch(s).
⏹️ Early stopping triggered after 55 epochs.
